# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv"
df = pd.read_csv(url)

print(f"Initial Dataset Shape: {df.shape}")


if 'children' in df.columns:
    df['children'] = df['children'].fillna(0)
if 'country' in df.columns:
    df['country'] = df['country'].fillna('Unknown')
if 'agent' in df.columns:
    df['agent'] = df['agent'].fillna('No Agent')
if 'company' in df.columns:
    df['company'] = df['company'].fillna('No Company')


leaky_cols = ['reservation_status', 'reservation_status_date']
df = df.drop(columns=[col for col in leaky_cols if col in df.columns])


y = df['is_canceled']
X = df.drop(columns=['is_canceled'])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)


X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(f"Encoded Training Features Shape: {X_train_encoded.shape}")
print(f"Encoded Testing Features Shape: {X_test_encoded.shape}")

Initial Dataset Shape: (9404, 32)
Encoded Training Features Shape: (6582, 330)
Encoded Testing Features Shape: (2822, 330)


### ✍️ Your Response:
1. The initial dataset has 9404 rows and 32 columns.
2. It contains a mix of categorical features like hotel type, month, meal, and customer type, alongside numerical features like lead time, stay duration, number of guests, and pricing.
3. We cleaned the data by imputing missing values in children, country, agent, and company columns. We also dropped target-leaking reservation status columns, split the data into 70% training and 30% testing sets to prevent any leakage, and then applied one-hot encoding to convert categorical variables into binary columns.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [12]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


nb_model = GaussianNB()


nb_model.fit(X_train_encoded, y_train)


y_pred_nb = nb_model.predict(X_test_encoded)


nb_accuracy = accuracy_score(y_test, y_pred_nb)
print("=== Naïve Bayes Model Performance ===")
print(f"Accuracy: {nb_accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_nb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

=== Naïve Bayes Model Performance ===
Accuracy: 0.5712

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.43      0.60      2121
           1       0.37      0.98      0.53       701

    accuracy                           0.57      2822
   macro avg       0.68      0.71      0.57      2822
weighted avg       0.83      0.57      0.59      2822

Confusion Matrix:
[[ 922 1199]
 [  11  690]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response:
1. The model has a low overall accuracy of 57.12%. This means the model is excellent at capturing almost all actual cancellations, even though it flags many non-cancellations as canceled. Depending on the business goal, recall or f1-score is the best metric to judge performance.
2. This model is highly useful for risk mitigation and worst-case scenario planning. Because it flags nearly every potential cancellation, the hotel can use it for real-time overbooking alerts or to trigger automated follow-up emails asking guests to confirm their reservations, helping to secure revenue before a potential no-show.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)


svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)


y_pred_svm = svm_model.predict(X_test_scaled)


svm_accuracy = accuracy_score(y_test, y_pred_svm)
print("=== Support Vector Machine Model Performance ===")
print(f"Accuracy: {svm_accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_svm))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

=== Support Vector Machine Model Performance ===
Accuracy: 0.9118

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      2121
           1       0.82      0.83      0.82       701

    accuracy                           0.91      2822
   macro avg       0.88      0.88      0.88      2822
weighted avg       0.91      0.91      0.91      2822

Confusion Matrix:
[[1993  128]
 [ 121  580]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response:
1. The SVM model performs exceptionally well, achieving an accuracy of 91.18% on the test set. Precision for predictions of canceled bookings is 82% and recall is 83%. The overall f1-score of 91% is the best metric to judge performance because it balances both false positives and false negatives to confirm model stability.
2. SVM excels in complex scenarios where features have non-linear or multi-dimensional relationships that simpler linear classifiers cannot capture. It is highly valuable when predicting guest cancellations depends on a subtle combination of factors like lead time combined with pricing patterns and guest history.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [14]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_encoded, y_train, test_size=0.15, random_state=42, stratify=y_train
)


scaler_nn = StandardScaler()
X_train_sub_scaled = scaler_nn.fit_transform(X_train_sub)
X_val_scaled = scaler_nn.transform(X_val)
X_test_scaled_nn = scaler_nn.transform(X_test_encoded)


mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15
)


mlp_model.fit(X_train_sub_scaled, y_train_sub)


y_pred_nn = mlp_model.predict(X_test_scaled_nn)


nn_accuracy = accuracy_score(y_test, y_pred_nn)
print("=== Neural Network Model Performance ===")
print(f"Accuracy: {nn_accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_nn))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nn))

=== Neural Network Model Performance ===
Accuracy: 0.9125

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      2121
           1       0.83      0.82      0.82       701

    accuracy                           0.91      2822
   macro avg       0.88      0.88      0.88      2822
weighted avg       0.91      0.91      0.91      2822

Confusion Matrix:
[[2001  120]
 [ 127  574]]


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response:
This neural network model achieves a high accuracy of 91.25 percent, making it slightly more accurate than the support vector machine and significantly stronger than the baseline Naïve Bayes model. It balances precision and recall well, showing highly consistent performance across both canceled and non-canceled bookings.

Whether the business would comfortable with a black box approach depends on who is using it. If the goal is simply to automate overbooking numbers behind the scenes, the high accuracy justifies its use. However, if hotel management wants to explain to operations or individual guests why specific bookings are flagged, this model is harder to justify because it lacks the simple, interpretable decision logic of traditional approaches.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [15]:
print("=== Model Accuracy Comparison ===")
print(f"Naïve Bayes Accuracy: {nb_accuracy:.4f}")
print(f"SVM Accuracy:         {svm_accuracy:.4f}")
print(f"Neural Network Accuracy: {nn_accuracy:.4f}\n")

print("=== Summary ===")
print("The Neural Network model achieved the highest accuracy at 91.25%, slightly outperforming the Support Vector Machine (91.18%). Both of these models far exceeded the baseline Naïve Bayes classifier, which had an accuracy of only 57.12%.")

=== Model Accuracy Comparison ===
Naïve Bayes Accuracy: 0.5712
SVM Accuracy:         0.9118
Neural Network Accuracy: 0.9125

=== Summary ===
The Neural Network model achieved the highest accuracy at 91.25%, slightly outperforming the Support Vector Machine (91.18%). Both of these models far exceeded the baseline Naïve Bayes classifier, which had an accuracy of only 57.12%.


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response:
The neural network achieved the highest overall accuracy at 91.25 percent, closely followed by the support vector machine at 91.18 percent. When it comes to training time, the Naïve Bayes model is incredibly fast but lacks the predictive accuracy needed for precise decision-making. In terms of ease of use and interpretability, the Naïve Bayes model is the most straightforward, while the neural network remains a complex black box that is difficult to explain to business stakeholders.

I recommend deploying the support vector machine model for this hotel business application. Even though the neural network has a tiny edge in accuracy, the support vector machine is marginally simpler to train and delivers virtually identical predictive performance. It provides a robust and reliable prediction of customer cancellations, giving hotel operations the confidence they need to optimize booking thresholds and mitigate risk without unnecessary computational complexity.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response:
I recommend deploying the support vector machine model to predict hotel cancellations. It delivers an outstanding 91.18 percent accuracy, which is virtually identical to the more complex neural network, but runs faster and is easier to configure. This model helps the hotel optimize staffing and rooms by accurately predicting booking cancellations, which minimizes lost revenue from no-shows and avoids overbooking complaints. A potential risk is that customer cancellation habits can change over time, so we must monitor performance. Adding local weather forecasts and regional event calendars to our data in the future would likely improve predictive power even further.

This project directly supports my customized learning outcome of mastering practical predictive modeling in a commercial setting. Working on this dataset allowed me to build, tune, and evaluate multiple algorithms using real business constraints, demonstrating how to balance raw classification accuracy against operational needs like model explainability and training efficiency.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [16]:
!jupyter nbconvert --to html "/content/assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook /content/assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 336462 bytes to /content/assignment_12_bayes_svm_neural.html
